# E3 — Chính quy hóa: đo riêng và đo kết hợp (yêu cầu 5.1.3)

**Hướng dẫn người chạy:** bật GPU **T4** (không chọn P100) và Internet → sửa `OWNER` → *Save & Run All* (chạy nền) → tải `outputs/` (file `.zip` in ở cuối) gửi cho điều phối. Không sửa gì khác.

**Lưu ý:** notebook này phải chạy **cả hai seed trong cùng một phiên** (p\* cần trung bình 2 seed). 10 cấu hình × 2 seed = 20 lượt.

Tham chiếu của riêng E3 là **R0 = không chính quy hóa** (không weight decay, không dropout, không augmentation). Phần còn lại giữ như công thức mặc định (SGD momentum, lr 0.1, cosine, batch 128, 20 epoch).

| config | WD | Dropout | Aug |
|---|---|---:|---|
| E3-R0 | 0 | 0 | không |
| E3-R1-wd | 5e-4 | 0 | không |
| E3-R2-do0.1 / R3-do0.3 / R4-do0.5 | 0 | 0.1 / 0.3 / 0.5 | không |
| E3-R5-flip | 0 | 0 | flip |
| E3-R6-crop | 0 | 0 | crop (pad 4) |
| E3-R7-jitter | 0 | 0 | color jitter |
| E3-R8-augall | 0 | 0 | flip + crop + jitter |
| E3-C-combined | 5e-4 | **p\*** | flip + crop + jitter |

## Lý do các lựa chọn chính

- **Dropout đặt *trong* mỗi BasicBlock** (giữa conv1 và conv2, kiểu Wide-ResNet). Đặt trước lớp FC cuối thì tác động quá nhỏ để đo được; ResNet-18 ít tham số dư ở lớp FC (256×10).
- **Vì sao có train-eval:** train acc đo *trong lúc huấn luyện* bị méo vì augmentation và dropout đang bật (aug mạnh làm train acc thấp giả tạo, khiến khoảng cách train–test trông nhỏ giả tạo). Vì vậy E3 đo thêm acc/loss trên 5 000 ảnh train cố định, ở eval mode, không aug, để có `gen_gap = train_eval_acc − test_acc` so sánh công bằng giữa các cấu hình. E1/E2/E4 không đo.
- **Vì sao p\* chọn theo val chứ không theo test:** test chỉ để báo cáo; chọn siêu tham số trên test là rò rỉ. p\* = mức dropout có `final_val_acc` trung bình 2 seed cao nhất trong R2–R4. **Hạn chế:** p\* được chọn khi chưa có augmentation.
- **Color jitter:** brightness/contrast/saturation = 0.4, không dùng hue; áp trước normalize.
- **Early stopping (hậu kiểm):** mọi lượt luôn huấn luyện đủ 20 epoch và log val/test mỗi epoch; ES = theo dõi val loss, patience 5, lấy mô hình tốt nhất tới lúc dừng. Kết quả y hệt dừng thật nhưng không phải chạy lại. "ES riêng" đọc ở R0, "ES kết hợp" đọc ở E3-C-combined.

In [ ]:
# ===== CHỈ SỬA CÁC DÒNG TRONG KHỐI NÀY =====
OWNER = "ten_ban"          # tên của bạn (không dấu, không khoảng trắng)
SEEDS = [0, 1]    # phải là [0, 1]: p* cần trung bình 2 seed
# ============================================

import os, sys
REPO_URL = "https://github.com/minh301003/Deep-Learning-Project-2026B.git"
TAG = "v1"                 # phiên bản code được ghim; điều phối tạo tag này trước khi mọi người chạy
if not os.path.exists("repo"):
    !git clone --depth 1 --branch $TAG $REPO_URL repo
sys.path.insert(0, "repo")
import pandas as pd
import btl
btl.init(owner=OWNER)

In [ ]:
assert SEEDS == [0, 1], "Notebook 03 phải chạy cả hai seed trong cùng một phiên"

NONE = dict(weight_decay=0, aug_crop=False, aug_flip=False, eval_train=True)     # R0: không chính quy hóa
RUNS = {
    "E3-R0":        NONE,
    "E3-R1-wd":     {**NONE, "weight_decay": 5e-4},
    "E3-R2-do0.1":  {**NONE, "dropout": 0.1},
    "E3-R3-do0.3":  {**NONE, "dropout": 0.3},
    "E3-R4-do0.5":  {**NONE, "dropout": 0.5},
    "E3-R5-flip":   {**NONE, "aug_flip": True},
    "E3-R6-crop":   {**NONE, "aug_crop": True},
    "E3-R7-jitter": {**NONE, "aug_jitter": True},
    "E3-R8-augall": {**NONE, "aug_flip": True, "aug_crop": True, "aug_jitter": True},
}

for seed in SEEDS:
    for name, overrides in RUNS.items():
        btl.run(name, seed, **overrides)

In [ ]:
# Chọn p* theo final_val_acc trung bình 2 seed (KHÔNG dùng test), rồi chạy cấu hình kết hợp
runs = pd.read_csv(f"{btl.OUT_DIR}/runs.csv")
dropout_cfgs = {"E3-R2-do0.1": 0.1, "E3-R3-do0.3": 0.3, "E3-R4-do0.5": 0.5}
dropout = runs[runs.config.isin(dropout_cfgs) & (runs.status == "completed")]
val_by_p = dropout.groupby("config").final_val_acc.mean()
val_by_p.to_csv(f"{btl.OUT_DIR}/p_star.csv")
print(val_by_p)
p_star = dropout_cfgs[val_by_p.idxmax()]
print("p* =", p_star)

for seed in SEEDS:
    btl.run("E3-C-combined", seed, weight_decay=5e-4, dropout=p_star, aug_jitter=True, eval_train=True)

In [ ]:
runs = pd.read_csv(f"{btl.OUT_DIR}/runs.csv")
print(runs[["config", "seed", "status", "final_test_acc", "duration_min"]].to_string(index=False))
bad = runs[runs.status != "completed"]
print(f"\nTổng {len(runs)} lượt, {len(bad)} lượt diverged/crashed" + (" -> ĐỌC KỸ cột notes trong runs.csv" if len(bad) else ""))
btl.zip_outputs("03_regularization")   # gửi file .zip này cho điều phối